# Convert Markdown to interleaved Parquet

This tutorial loads 1,000 Markdown documents from [PIN-14M](https://huggingface.co/datasets/m-a-p/PIN-14M), converts their text and image references into interleaved rows on Ray, materializes the image bytes while writing Parquet, and reads one output shard back for inspection.

Use Python 3.11–3.13 and install the required packages before running the notebook:

```bash
pip install "nemo_curator[interleaved_cpu]"
```

In [ ]:
from io import BytesIO

import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from IPython.display import Code, Markdown
from IPython.display import display as ipy_display
from PIL import Image

from nemo_curator.backends.ray_data import RayDataExecutor
from nemo_curator.core.client import RayClient
from nemo_curator.pipeline import Pipeline
from nemo_curator.stages.interleaved import MarkdownToInterleavedStage
from nemo_curator.stages.interleaved.io.readers.parquet import InterleavedParquetReaderStage
from nemo_curator.stages.interleaved.io.writers.tabular import InterleavedParquetWriterStage
from nemo_curator.tasks import DocumentBatch, FileGroupTask

NUM_ROWS = 1_000
BATCH_SIZE = 250
OUTPUT_PATH = "pin14m_interleaved"

## Load and inspect the raw Markdown

Streaming limits the download to the rows used by this tutorial. The first three documents are shown as safe Markdown source; their relative image references are materialized during conversion below.

In [ ]:
dataset = load_dataset("m-a-p/PIN-14M", "pin", split="train", streaming=True)
raw_df = pd.DataFrame(list(dataset.take(NUM_ROWS)))
for row in raw_df.head(3).itertuples(index=False):
    ipy_display(Markdown(f"### Raw document `{row.id}`"))
    ipy_display(Code(row.md, language="markdown"))

## Convert and write

`MarkdownToInterleavedStage` preserves document metadata and emits text and lazy image-reference rows in reading order. The input is split into 250-document tasks so Ray can process multiple batches. Downloading `content_image.tar.gz` once avoids repeatedly scanning the large compressed archive over HTTP. `materialize_on_write=True` then resolves each image before Parquet is written; the default `on_materialize_error="error"` stops the run instead of silently writing missing image bytes.

In [ ]:
image_archive = hf_hub_download(
    repo_id="m-a-p/PIN-14M",
    repo_type="dataset",
    filename="data/DocLayNet/content_image.tar.gz",
)
documents = [
    DocumentBatch(
        dataset_name="PIN-14M",
        data=raw_df.iloc[start : start + BATCH_SIZE],
        _metadata={"source_files": [f"hf://datasets/m-a-p/PIN-14M#rows={start}:{min(start + BATCH_SIZE, len(raw_df))}"]},
    )
    for start in range(0, len(raw_df), BATCH_SIZE)
]

pipeline = Pipeline(
    name="markdown_to_interleaved",
    stages=[
        MarkdownToInterleavedStage(image_source_uri=image_archive),
        InterleavedParquetWriterStage(
            path=OUTPUT_PATH,
            materialize_on_write=True,
            mode="overwrite",
        ),
    ],
)

with RayClient():
    written = pipeline.run(executor=RayDataExecutor(), initial_tasks=documents)

parquet_paths = [path for task in written for path in task.data]
print("Wrote:\n" + "\n".join(parquet_paths))

## Read and inspect the interleaved data

Read one shard back through Curator, then render the first three documents with their text and materialized images in reading order. Metadata rows are omitted from the display.

In [ ]:
file_task = FileGroupTask(dataset_name="PIN-14M", data=[parquet_paths[0]], _metadata={})
interleaved = InterleavedParquetReaderStage().process(file_task)
interleaved_df = interleaved.to_pandas()

for sample_id in interleaved_df["sample_id"].drop_duplicates().head(3):
    ipy_display(Markdown(f"### Interleaved document `{sample_id}`"))
    rows = interleaved_df[interleaved_df["sample_id"] == sample_id].sort_values("position")
    for row in rows.itertuples(index=False):
        if row.modality == "text":
            print(row.text_content)
        elif row.modality == "image":
            with Image.open(BytesIO(row.binary_content)) as image:
                image.thumbnail((640, 480))
                ipy_display(image.copy())